In [1]:
#Importations et configuration de l'environnement
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import optuna
from sklearn.utils.class_weight import compute_class_weight
import warnings
from Orchestrator import DOSSIER_RACINE, DOSSIER_TF, DOSSIER_MODELES, DOSSIER_LOGS, DOSSIER_TRANSFORMED
warnings.filterwarnings('ignore')

I0000 00:00:1775251759.236079   15646 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775251759.281827   15646 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1775251761.325785   15646 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


/home/clement/ProjetML/RN_test/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Rechargement des pipelines de données
dossier_tf = os.path.join(DOSSIER_TF, 'data_tf_pipelines')

train_ds = tf.data.Dataset.load(os.path.join(DOSSIER_TF, 'train'))
val_ds = tf.data.Dataset.load(os.path.join(DOSSIER_TF, 'val'))
test_ds = tf.data.Dataset.load(os.path.join(DOSSIER_TF, 'test'))

def valider_et_extraire_dimension(dataset):
    """Vérifie la structure du dataset et extrait dynamiquement le nombre de features."""
    element_spec = dataset.element_spec
    shape_x = element_spec[0].shape
    shape_y = element_spec[1].shape
    
    assert len(shape_x) == 2, "La matrice de caractéristiques doit être 2D (Batch, Features)."
    assert shape_y == (None,), "Le vecteur cible doit être 1D."
    
    nb_features = shape_x[1]
    assert nb_features > 0, "Le nombre de caractéristiques doit être strictement positif."
    
    print(f"Validation réussie. Dimension d'entrée détectée : {nb_features} caractéristiques.")
    return nb_features

NB_FEATURES = valider_et_extraire_dimension(train_ds)

Validation réussie. Dimension d'entrée détectée : 31 caractéristiques.


E0000 00:00:1775251762.978961   15646 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1775251762.979402   15717 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1775251762.995841   15646 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
def calculer_poids_classes(dataset):
    """Extrait les étiquettes et calcule les poids pour pénaliser les erreurs sur la classe minoritaire."""
    # Aplatissement explicite pour s'assurer d'un tableau 1D
    y_true = np.concatenate([y.numpy().flatten() for x, y in dataset])
    classes = np.unique(y_true)
    poids = compute_class_weight(class_weight='balanced', classes=classes, y=y_true)
    
    # Cast explicite en int et float natifs exigé par Keras pour l'association des poids
    class_weights_dict = {int(c): float(p) for c, p in zip(classes, poids)}
    return class_weights_dict, y_true

class_weights, y_train_labels = calculer_poids_classes(train_ds)

def valider_poids(poids_dict, labels):
    """Vérifie que la classe minoritaire reçoit un poids mathématiquement supérieur."""
    ratio_defaut = np.sum(labels == 1) / len(labels)
    assert 0 in poids_dict and 1 in poids_dict, "Les deux classes (0 et 1) doivent avoir un poids."
    
    if ratio_defaut < 0.5:
        assert poids_dict[1] > poids_dict[0], "Erreur : La classe minoritaire (1) n'est pas suffisamment pénalisée."
    print("Validation des poids réussie :", poids_dict)

valider_poids(class_weights, y_train_labels)

Validation des poids réussie : {0: 0.5656922755556119, 1: 4.305622470610908}


In [4]:
def creer_baseline(nb_features):
    """Construit un Perceptron Multicouche simple avec Batch Normalization et Dropout."""
    entree = keras.Input(shape=(nb_features,))
    
    x = keras.layers.Dense(64, activation='relu')(entree)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.3)(x)
    
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.2)(x)
    
    sortie = keras.layers.Dense(1, activation='sigmoid')(x)
    
    modele = keras.Model(inputs=entree, outputs=sortie, name="Baseline_MLP")
    
    metriques = [
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.AUC(name='auc', curve='PR')
    ]
    
    modele.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=metriques
    )
    return modele

def valider_architecture_baseline():
    """Test unitaire : vérifie la compilation du graphe et la validité de la fonction de sortie."""
    modele = creer_baseline(NB_FEATURES)
    tenseur_test = tf.random.normal((1, NB_FEATURES))
    prediction = modele(tenseur_test)
    
    assert prediction.shape == (1, 1), f"Erreur de dimension de sortie : {prediction.shape}"
    assert 0.0 <= float(prediction.numpy()[0][0]) <= 1.0, "Erreur : La sortie sigmoïde doit être comprise entre 0 et 1."
    assert isinstance(modele.loss, keras.losses.BinaryCrossentropy), "Erreur : La fonction de perte n'est pas BinaryCrossentropy."
    
    print("Validation de l'architecture réussie. Le graphe compile correctement.")

valider_architecture_baseline()

Validation de l'architecture réussie. Le graphe compile correctement.


In [5]:
def obtenir_callbacks(nom_experience):
    """Configure les callbacks pour la sauvegarde et l'arrêt prématuré."""
    chemin_modele = os.path.join(DOSSIER_MODELES, f"{nom_experience}.keras")
    chemin_log = os.path.join(DOSSIER_LOGS, nom_experience)
    
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_recall', mode='max', patience=10, restore_best_weights=True, verbose=1
    )
    
    model_checkpoint = keras.callbacks.ModelCheckpoint(
        filepath=chemin_modele, monitor='val_recall', mode='max', save_best_only=True, verbose=0
    )
    
    tensorboard = keras.callbacks.TensorBoard(log_dir=chemin_log, histogram_freq=1)
    
    return [early_stopping, model_checkpoint, tensorboard]

def valider_callbacks():
    """Vérifie l'instanciation correcte des objets callbacks."""
    cb_list = obtenir_callbacks("test_cb")
    assert any(isinstance(cb, keras.callbacks.EarlyStopping) for cb in cb_list), "EarlyStopping manquant."
    assert any(isinstance(cb, keras.callbacks.ModelCheckpoint) for cb in cb_list), "ModelCheckpoint manquant."
    print("Validation des callbacks réussie.")

valider_callbacks()

Validation des callbacks réussie.


In [6]:
def entrainer_baseline():
    """Encapsule l'entraînement du modèle de référence."""
    baseline_model = creer_baseline(NB_FEATURES)
    callbacks_baseline = obtenir_callbacks("baseline_mlp")

    print("Début de l'entraînement de la Baseline...")
    historique = baseline_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=50,
        class_weight=class_weights,
        callbacks=callbacks_baseline,
        verbose=1
    )
    return baseline_model

baseline_model_entraine = entrainer_baseline()

def valider_inference(modele, dataset_test):
    """Test unitaire post-entraînement : vérifie que le modèle produit des probabilités valides sur un lot de test."""
    for x_batch, y_batch in dataset_test.take(1):
        predictions = modele.predict(x_batch, verbose=0)
        assert predictions.shape == (x_batch.shape[0], 1), "Dimension des prédictions incorrecte."
        assert np.all((predictions >= 0.0) & (predictions <= 1.0)), "Des probabilités hors de l'intervalle [0, 1] ont été détectées."
    print("Validation d'inférence réussie. Le modèle prédit correctement sur des données non vues.")

valider_inference(baseline_model_entraine, test_ds)

Début de l'entraînement de la Baseline...
Epoch 1/50


  1/699 ━━━━━━━━━━━━━━━━━━━━ 20:10 2s/step - auc: 0.0940 - loss: 0.7095 - precision: 0.0744 - recall: 0.4500

 15/699 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - auc: 0.1112 - loss: 0.8320 - precision: 0.1076 - recall: 0.4869  

 32/699 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.1234 - loss: 0.8222 - precision: 0.1197 - recall: 0.5217

 49/699 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.1333 - loss: 0.8091 - precision: 0.1277 - recall: 0.5450

...

690/699 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - auc: 0.3135 - loss: 0.5891 - precision: 0.2261 - recall: 0.6927

699/699 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - auc: 0.3112 - loss: 0.5880 - precision: 0.2241 - recall: 0.6894 - val_auc: 0.3164 - val_loss: 0.5808 - val_precision: 0.2247 - val_recall: 0.6799


Epoch 11: early stopping


Restoring model weights from the end of the best epoch: 1.


Validation d'inférence réussie. Le modèle prédit correctement sur des données non vues.


In [7]:
def objective(trial):
    """Fonction objective pour l'optimisation des hyperparamètres."""
    keras.backend.clear_session()
    
    entree = keras.Input(shape=(NB_FEATURES,))
    
    nb_couches = trial.suggest_int("nb_couches", 1, 3)
    taux_apprentissage = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    
    x = entree
    for i in range(nb_couches):
        unites = trial.suggest_int(f"unites_couche_{i}", 32, 128, step=32)
        dropout = trial.suggest_float(f"dropout_couche_{i}", 0.1, 0.5)
        
        x = keras.layers.Dense(unites, activation='relu')(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Dropout(dropout)(x)
        
    sortie = keras.layers.Dense(1, activation='sigmoid')(x)
    modele = keras.Model(inputs=entree, outputs=sortie)
    
    modele.compile(
        optimizer=keras.optimizers.Adam(learning_rate=taux_apprentissage),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.Recall(name='recall')]
    )
    
    try:
        history = modele.fit(
            train_ds,
            validation_data=val_ds,
            epochs=15, 
            class_weight=class_weights,
            verbose=0
        )
    except Exception as e:
        # Permet à Optuna de continuer en cas d'erreur ponctuelle de calcul de gradient
        raise optuna.exceptions.TrialPruned() from e
    
    val_recall_max = max(history.history['val_recall'])
    return val_recall_max

def valider_et_lancer_optuna():
    """Valide l'initialisation de l'étude et lance la recherche."""
    tf.get_logger().setLevel('ERROR') 
    
    etude = optuna.create_study(direction="maximize", study_name="Optimisation_MLP_Defaut")
    print("Lancement de l'étude Optuna (3 essais de test)...")
    etude.optimize(objective, n_trials=3) 
    
    assert len(etude.trials) > 0, "L'étude Optuna n'a enregistré aucun essai."
    print("Validation Optuna réussie. Meilleurs hyperparamètres :", etude.best_params)

valider_et_lancer_optuna()

[I 2026-04-03 23:30:32,696] A new study created in memory with name: Optimisation_MLP_Defaut


Lancement de l'étude Optuna (3 essais de test)...


[I 2026-04-03 23:30:58,962] Trial 0 finished with value: 0.6821757555007935 and parameters: {'nb_couches': 1, 'learning_rate': 0.0006619724744197174, 'unites_couche_0': 64, 'dropout_couche_0': 0.3301153922172766}. Best is trial 0 with value: 0.6821757555007935.


[I 2026-04-03 23:31:29,106] Trial 1 finished with value: 0.6832996010780334 and parameters: {'nb_couches': 1, 'learning_rate': 0.0021627988527678755, 'unites_couche_0': 64, 'dropout_couche_0': 0.44865152644986517}. Best is trial 1 with value: 0.6832996010780334.


[I 2026-04-03 23:32:06,284] Trial 2 finished with value: 0.6961115002632141 and parameters: {'nb_couches': 1, 'learning_rate': 0.00012001224070180619, 'unites_couche_0': 96, 'dropout_couche_0': 0.3748587244957742}. Best is trial 2 with value: 0.6961115002632141.


Validation Optuna réussie. Meilleurs hyperparamètres : {'nb_couches': 1, 'learning_rate': 0.00012001224070180619, 'unites_couche_0': 96, 'dropout_couche_0': 0.3748587244957742}
